In [1]:
import json
from transformers import AutoTokenizer
from tqdm import tqdm
import os
from tqdm import tqdm
from openai import OpenAI
# Настройки локального API-сервера
client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")
import time
from openai import OpenAI

/home/vitalii/llm_project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#    dataset/master_dataset_clean.jsonl - 

#
#

def split_dataset():
    # Указываем пути к файлам
    input_file = "dataset/master_dataset_clean.jsonl"
    out_bucket_1 = "dataset/distill/tales_bucket1_ideal.jsonl"     # <= 1536
    out_bucket_2 = "dataset/distill/tales_bucket2_condense.jsonl"  # 1537 - 4096
    out_bucket_3 = "dataset/distill/tales_bucket3_drop.jsonl"      # > 4096

    # Загружаем токенизатор целевой модели
    print("Загрузка токенизатора...")
    tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-2.8b")

    # Списки для хранения данных
    bucket_1, bucket_2, bucket_3 = [], [], []

    print("Анализ датасета...")
    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
        for line in tqdm(lines, desc="Подсчет токенов"):
            if not line.strip():
                continue
                
            data = json.loads(line)
            # Предполагается, что текст сказки лежит в ключе 'text'
            text = data.get("completion", "") 
            
            # Считаем токены
            token_count = len(tokenizer.encode(text))

            if token_count <= 1536:
                bucket_1.append(data)
            elif token_count <= 4096:
                bucket_2.append(data)
            else:
                bucket_3.append(data)

    # Функция для сохранения JSONL
    def save_jsonl(data_list, filename):
        with open(filename, "w", encoding="utf-8") as f:
            for item in data_list:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    # Сохраняем результаты
    print(f"\nСохранение результатов...")
    save_jsonl(bucket_1, out_bucket_1)
    save_jsonl(bucket_2, out_bucket_2)
    save_jsonl(bucket_3, out_bucket_3)

    print("-" * 30)
    print("Готово! Статистика разбивки:")
    print(f"Корзина 1 (Идеальные, <= 1536): {len(bucket_1)} шт.")
    print(f"Корзина 2 (На сжатие, 1537-4096): {len(bucket_2)} шт.")
    print(f"Корзина 3 (Исключены, > 4096): {len(bucket_3)} шт.")
    print("-" * 30)

if __name__ == "__main__":
    split_dataset()

Загрузка токенизатора...
Анализ датасета...


Подсчет токенов: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4355/4355 [00:16<00:00, 265.56it/s]



Сохранение результатов...
------------------------------
Готово! Статистика разбивки:
Корзина 1 (Идеальные, <= 1536): 1936 шт.
Корзина 2 (На сжатие, 1537-4096): 1745 шт.
Корзина 3 (Исключены, > 4096): 674 шт.
------------------------------


In [ ]:
SYSTEM_PROMPT = """You are an expert folklorist and a conservative editor of authentic 18th-to-19th-century folklore. Your task is to rewrite and logically repair the provided fairy tale to be strictly between 900 and 1000 words.

Follow these rules strictly:
1. NO CENSORSHIP: Do not soften or "Disneyfy" the text. Preserve all dark, macabre, or violent elements typical of original folklore.
2. PRESERVE DETAIL: Do NOT write a short summary. Preserve enough descriptive detail, dialogue, and secondary characters to ensure the story is rich and full, aiming for the 900-1000 word target, but do NOT exceed 1000 words.
3. LOGICAL REPAIR: Fix any broken plots or sudden jumps, but keep the exact pacing of a traditional fairy tale.
4. TONE: Maintain an archaic, objective narrative tone. Do not add artificial moral lessons.

Output ONLY the story text. Do not add any introductory or concluding remarks."""

def run_benchmark():
    # Читаем первую строчку из корзины на сжатие
    with open("dataset/distill/tales_bucket2_condense.jsonl", "r", encoding="utf-8") as f:
        first_line = f.readline()
        data = json.loads(first_line)
        original_text = data.get("completion", "")

    print(f"Длина оригинала: {len(original_text.split())} слов.")
    print("Отправка запроса в Qwen 32B... Ждем генерацию...\n")

    start_time = time.time()

    # Делаем вызов к локальной модели
    response = client.chat.completions.create(
        model="local-model",  # Имя игнорируется локальным сервером
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Story:\n{original_text}"}
        ],
        temperature=0.3, # Низкая температура, чтобы модель не фантазировала, а работала с текстом
        max_tokens=2000  # С запасом под наш лимит слов
    )

    end_time = time.time()
    
    result_text = response.choices[0].message.content
    duration = end_time - start_time
    
    # Считаем слова и оцениваем токены (в английском ~1.3 токена на слово)
    word_count = len(result_text.split())
    approx_tokens = word_count * 1.3
    
    print("-" * 50)
    print(f"Результат (первые 300 символов):\n{result_text[:300]}...\n")
    print("-" * 50)
    print(f"Время выполнения: {duration:.2f} секунд")
    print(f"Длина результата: {word_count} слов")
    
    if duration > 0:
        tps = approx_tokens / duration
        print(f"Примерная скорость: {tps:.1f} токенов в секунду")

if __name__ == "__main__":
    run_benchmark()

In [3]:
# прогоняем через локальную модель QWEN2.5 32b для сжатия (1200-1400 слов) и починки возможных смысловых ошибок
# tales_bucket2_condense.jsonl -> tales_bucket2_processed.jsonl
# tales_bucket3_drop.jsonl -> tales_bucket3_processed.jsonl

SYSTEM_PROMPT = """You are an expert folklorist and a master storyteller rewriting authentic 18th-to-19th-century folklore. Your task is to rewrite and logically repair the provided fairy tale, creating a rich, detailed, and immersive narrative of about 1200 to 1400 words.

Follow these rules strictly:
1. EXPAND AND ENRICH: Do not just summarize the events. Actively expand the narrative. Add rich sensory details to the environments, flesh out the atmosphere, and expand the dialogues. Make the scenes vivid and descriptive while keeping the original plot intact.
2. NO CENSORSHIP: Do not soften or "Disneyfy" the text. Preserve all dark, macabre, or violent elements typical of original folklore.
3. LOGICAL REPAIR: Fix any broken plots or sudden jumps, ensuring smooth, detailed transitions between expanded scenes.
4. TONE: Maintain an archaic, objective narrative tone (e.g., Brothers Grimm style). Do not add artificial moral lessons or modern empathy.

Output ONLY the story text. Do not add any introductory or concluding remarks."""

def batch_process(input_file, output_file):
    
    # --- ШАГ 1: Защита от сбоев (Вспоминаем, где остановились) ---
    processed_count = 0
    
    # Если файл с результатами уже существует, значит скрипт раньше запускали
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            # Считаем количество строк. 1 строка = 1 готовая сказка.
            processed_count = sum(1 for line in f)
        print(f"Найдено {processed_count} готовых сказок. Продолжаем работу...")

    # --- ШАГ 2: Узнаем общий объем работы ---
    # Считаем все строки в исходном датасете, чтобы прогресс-бар понимал масштаб
    with open(input_file, "r", encoding="utf-8") as f:
        total_tales = sum(1 for line in f)
    
    if processed_count >= total_tales:
        print("Все сказки уже обработаны!")
        return

    print(f"Осталось обработать: {total_tales - processed_count} из {total_tales}")

    # --- ШАГ 3: Основной цикл инференса ---
    # Открываем исходный файл для чтения ("r")
    with open(input_file, "r", encoding="utf-8") as in_f:
        
        # Открываем файл результатов для ДОЗАПИСИ ("a" - append). 
        # Мы ничего не удаляем, только добавляем новые строки в конец.
        with open(output_file, "a", encoding="utf-8") as out_f:
            
            # Настраиваем прогресс-бар
            progress_bar = tqdm(total=total_tales, initial=processed_count, desc="Генерация Qwen 32B")
            
            # Идем по каждой строке исходного файла по очереди
            for index, line in enumerate(in_f):
                
                # Если индекс этой сказки меньше, чем мы уже обработали, просто пропускаем её
                if index < processed_count:
                    continue
                
                # Распаковываем JSON-строку в словарь Python и достаем текст
                data = json.loads(line)
                original_text = data.get("completion", "")

                try:
                    # Отправляем текст в нейросеть
                    response = client.chat.completions.create(
                        model="local-model",
                        messages=[
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": f"Story:\n{original_text}"}
                        ],
                        temperature=0.3,
                        max_tokens=2048 
                    )
                    
                    # Забираем переписанный текст и заменяем им старый в словаре
                    result_text = response.choices[0].message.content
                    data["completion"] = result_text
                    word_count = len(result_text.split())
                    data["token_count"] = int(word_count * 1.3)
                    # --- ШАГ 4: Безопасное сохранение ---
                    # Превращаем словарь обратно в JSON и пишем в файл результатов
                    out_f.write(json.dumps(data, ensure_ascii=False) + "\n")
                    
                    # Принудительно сбрасываем данные из кэша системы на жесткий диск.
                    # Это спасает данные при отключении электричества или зависании.
                    out_f.flush() 

                    # Двигаем ползунок прогресс-бара
                    progress_bar.update(1)

                except Exception as e:
                    # Если отвалился сервер или API — ловим ошибку и красиво выходим
                    print(f"\nОшибка на сказке под индексом {index}: {e}")
                    print("Скрипт остановлен. При перезапуске он продолжит с этого же места.")
                    break 
            
            progress_bar.close()

# if __name__ == "__main__":
#     INPUT_FILE = "dataset/distill/tales_bucket2_condense.jsonl"
#     OUTPUT_FILE = "dataset/distill/tales_bucket2_processed.jsonl"
    
#     batch_process(INPUT_FILE, OUTPUT_FILE)
if __name__ == "__main__":
    # Запускаем корзину отбракованных сказок
    INPUT_FILE = "dataset/distill/tales_bucket3_drop.jsonl"
    OUTPUT_FILE = "dataset/distill/tales_bucket3_processed.jsonl"
    
    batch_process(INPUT_FILE, OUTPUT_FILE)

Найдено 344 готовых сказок. Продолжаем работу...
Осталось обработать: 329 из 673


Генерация Qwen 32B: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 673/673 [21:21:29<00:00, 233.71s/it]


In [2]:
# Исправление и починнка датасета в случае возможного нарушения структуры JSONL
# dataset/distill/tales_bucket2_processed.jsonl ->  dataset/distill/tales_bucket2_fixed.jsonl
# dataset/distill/tales_bucket3_processed.jsonl ->  dataset/distill/tales_bucket3_fixed.jsonl


# Для корзины 2
# INPUT_FILE = "dataset/distill/tales_bucket2_processed.jsonl"
# OUTPUT_FILE = "dataset/distill/tales_bucket2_fixed.jsonl"

# Для корзины 3
INPUT_FILE = "dataset/distill/tales_bucket3_processed.jsonl"
OUTPUT_FILE = "dataset/distill/tales_bucket3_fixed.jsonl"

valid_lines = 0
fixed_lines = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
    for i, line in enumerate(f_in):
        # Убираем лишние пробелы и пустые строки по краям
        line = line.strip()
        if not line:
            continue 
        
        try:
            # Пробуем прочитать строку как JSON
            json.loads(line)
            # Если ошибок нет, записываем в новый файл с правильным переносом
            f_out.write(line + '\n')
            valid_lines += 1
            
        except json.JSONDecodeError:
            # Если строка битая, проверяем, не склеились ли два объекта: }{
            if '}{' in line:
                print(f"Найдена склеенная строка на позиции {i+1}. Разделяем...")
                # Разрезаем склеенные объекты
                parts = line.replace('}{', '}\n{').split('\n')
                
                for part in parts:
                    try:
                        json.loads(part)
                        f_out.write(part + '\n')
                        fixed_lines += 1
                    except json.JSONDecodeError as e:
                        print(f"Не удалось спасти кусок строки {i+1}: {e}")
            else:
                print(f"Неизвестная ошибка в строке {i+1}. Строка пропущена.")

print("-" * 30)
print(f"Готово! Целых сказок: {valid_lines}.")
print(f"Спасенных после склейки: {fixed_lines}.")
print(f"Твой чистый датасет сохранен как: {OUTPUT_FILE}")

------------------------------
Готово! Целых сказок: 673.
Спасенных после склейки: 0.
Твой чистый датасет сохранен как: dataset/distill/tales_bucket3_fixed.jsonl


In [6]:
# склеим корзины 2 и 3 после починки
# Пути к нашим починенным корзинам
bucket2_file = "dataset/distill/tales_bucket2_fixed.jsonl"
# bucket3_file = "dataset/distill/tales_bucket3_fixed.jsonl" корзину 3 после локальной модели не берём
bucket3_file = "dataset/distill/tales_bucket3_deepseek_processed.jsonl" # результат облачной DeepSeek

merged_file = "dataset/distill/tales_merged_raw.jsonl"

print("Начинаем склейку Корзины 2 и Корзины 3...\n")

b2_lines = 0
b3_lines = 0

with open(merged_file, 'w', encoding='utf-8') as f_out:
    
    # Читаем и записываем вторую корзину
    if os.path.exists(bucket2_file):
        with open(bucket2_file, 'r', encoding='utf-8') as f_in:
            for line in f_in:
                if line.strip():  # Игнорируем пустые строки
                    f_out.write(line.strip() + '\n')
                    b2_lines += 1
        print(f"Добавлено {b2_lines} сказок из Корзины 2")
    else:
        print(f"ВНИМАНИЕ: Файл {bucket2_file} не найден!")

    # Читаем и записываем третью корзину
    if os.path.exists(bucket3_file):
        with open(bucket3_file, 'r', encoding='utf-8') as f_in:
            for line in f_in:
                if line.strip():
                    f_out.write(line.strip() + '\n')
                    b3_lines += 1
        print(f"Добавлено {b3_lines} сказок из Корзины 3")
    else:
        print(f"ВНИМАНИЕ: Файл {bucket3_file} не найден!")

print("-" * 40)
print("Склейка завершена!")
print(f"Всего сказок в объединенном файле: {b2_lines + b3_lines}")
print(f"Файл сохранен как: {merged_file}")

Начинаем склейку Корзины 2 и Корзины 3...

Добавлено 1744 сказок из Корзины 2
Добавлено 673 сказок из Корзины 3
----------------------------------------
Склейка завершена!
Всего сказок в объединенном файле: 2417
Файл сохранен как: dataset/distill/tales_merged_raw.jsonl


In [2]:
import json
import os
from tqdm import tqdm
from openai import OpenAI
from transformers import AutoTokenizer

client = OpenAI(
    base_url="https://api.deepseek.com", 
    api_key="sk-ff584649fc1e4ba6a40873ebb4db08f4", # <--- ВСТАВЬ СВОЙ КЛЮЧ ЗДЕСЬ
    
)

# ДОБАВЛЕНО: Загружаем токенизатор Pythia один раз до начала цикла
print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-2.8b")

# Строгий промпт-ограничитель
SYSTEM_PROMPT = """You are an expert folklorist and a conservative editor of authentic 18th-to-19th-century folklore. Your task is to rewrite and logically repair the provided fairy tale.

CRITICAL CONSTRAINT: 
You must be concise. The story MUST be complete and conclusively finished within 1200-1500 words. Do not ramble. 

Follow these rules strictly:
1. NO CENSORSHIP: Preserve all dark, macabre, or violent elements typical of original folklore.
2. DYNAMIC PACING: Do not over-describe scenery or stretch dialogues. Keep the plot moving forward dynamically like a traditional folktale. 
3. LOGICAL REPAIR: Fix any broken plots or sudden jumps, ensuring smooth transitions.
4. TONE: Maintain an archaic, objective narrative tone. 

Output ONLY the story text. End the text logically."""

def process_final_dataset(input_file, output_file):
    processed_count = 0
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            processed_count = sum(1 for _ in f)
        print(f"Найдено {processed_count} уже проверенных сказок. Продолжаем...")

    # Считаем только валидные JSON-строки
    total_valid_lines = 0
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                total_valid_lines += 1

    if processed_count >= total_valid_lines:
        print("Все сказки в объединенном файле уже проверены!")
        return

    print(f"Осталось проверить: {total_valid_lines - processed_count} из {total_valid_lines}")

    # Порог токенов (с запасом под системные теги Pythia)
    TOKEN_LIMIT = 1900
    rewritten_count = 0
    skipped_errors = 0

    with open(input_file, "r", encoding="utf-8") as in_f:
        with open(output_file, "a", encoding="utf-8") as out_f:
            
            progress_bar = tqdm(total=total_valid_lines, initial=processed_count, desc="Финальный проход")
            
            for index, line in enumerate(in_f):
                if index < processed_count:
                    continue
                
                line = line.strip()
                if not line:
                    continue
                
                try:
                    data = json.loads(line)
                except json.JSONDecodeError:
                    skipped_errors += 1
                    progress_bar.update(1)
                    continue

                text = data.get("completion", "")
                
                # ИСПРАВЛЕНО: Считаем точные токены оригинального текста через токенизатор
                exact_tokens = len(tokenizer.encode(text))

                if exact_tokens > TOKEN_LIMIT:
                    try:
                        response = client.chat.completions.create(
                            model="deepseek-v4-flash",
                            messages=[
                                {"role": "system", "content": SYSTEM_PROMPT},
                                {"role": "user", "content": f"Story to condense:\n{text}"}
                            ],
                            temperature=0.3,
                            max_tokens=2048 
                        )
                        
                        result_text = response.choices[0].message.content
                        data["completion"] = result_text
                        
                        ### исправить метод подсчёта токенов - ИСПРАВЛЕНО
                        data["token_count"] = len(tokenizer.encode(result_text))
                        rewritten_count += 1
                        
                    except Exception as e:
                        print(f"\nОшибка на индексе {index}: {e}")
                        print("Скрипт остановлен. При перезапуске он продолжит с этого же места.")
                        break
                else:
                    # ИСПРАВЛЕНО: Если текст вписывается в лимиты, сохраняем его точный токенаж
                    data["token_count"] = exact_tokens

                out_f.write(json.dumps(data, ensure_ascii=False) + "\n")
                out_f.flush() 
                progress_bar.update(1)

            progress_bar.close()
            print("-" * 30)
            print(f"Проверка завершена!")
            print(f"Переписано слишком длинных сказок: {rewritten_count}")
            if skipped_errors > 0:
                print(f"Пропущено битых строк (JSON Error): {skipped_errors}")
            print(f"Твой идеальный датасет готов: {output_file}")

if __name__ == "__main__":
    INPUT_FILE = "dataset/distill/tales_merged_raw.jsonl"
    OUTPUT_FILE = "dataset/distill/tales_golden_dataset.jsonl"
    
    process_final_dataset(INPUT_FILE, OUTPUT_FILE)

Загрузка токенизатора...
Найдено 869 уже проверенных сказок. Продолжаем...
Осталось проверить: 1548 из 2417


Финальный проход: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2417/2417 [4:27:21<00:00, 10.36s/it]

------------------------------
Проверка завершена!
Переписано слишком длинных сказок: 831
Твой идеальный датасет готов: dataset/distill/tales_golden_dataset.jsonl


In [3]:

# Пути к файлам
bucket1_file = "dataset/distill/tales_bucket1_ideal.jsonl"
golden_file = "dataset/distill/tales_golden_dataset.jsonl"
final_output_file = "dataset/master_dataset_ready.jsonl"

valid_lines = 0
fixed_lines = 0
error_lines = 0

print("Начинаем финальную сборку и очистку...\n")

# Функция для безопасного чтения, починки и записи
def process_and_write(input_path, output_handle, source_name):
    global valid_lines, fixed_lines, error_lines
    
    if not os.path.exists(input_path):
        print(f"ВНИМАНИЕ: Файл {input_path} не найден!")
        return
    
    print(f"Обработка: {source_name}...")
    with open(input_path, 'r', encoding='utf-8') as f_in:
        for i, line in enumerate(f_in):
            line = line.strip()
            if not line:
                continue
            
            try:
                # Пробуем распарсить JSON
                json.loads(line)
                output_handle.write(line + '\n')
                valid_lines += 1
                
            except json.JSONDecodeError:
                # Если сломалось, ищем типичную склейку
                if '}{' in line:
                    parts = line.replace('}{', '}\n{').split('\n')
                    for part in parts:
                        try:
                            json.loads(part)
                            output_handle.write(part + '\n')
                            fixed_lines += 1
                        except json.JSONDecodeError:
                            error_lines += 1
                else:
                    error_lines += 1

# 1. СОБИРАЕМ И ЧИСТИМ ДАТАСЕТ
with open(final_output_file, 'w', encoding='utf-8') as f_out:
    process_and_write(bucket1_file, f_out, "Корзина 1 (Оригинальные идеальные сказки)")
    process_and_write(golden_file, f_out, "Золотой датасет (Обработанные сказки из корзин 2 и 3)")

print("-" * 40)
print("Сборка завершена!")
print(f"Целых сказок без ошибок: {valid_lines}")
print(f"Спасенных после склейки: {fixed_lines}")
if error_lines > 0:
    print(f"Удалено невосстановимых строк: {error_lines}")
print(f"Итоговый чистый файл: {final_output_file}\n")


# 2. ВЫВОДИМ СТАТИСТИКУ
def print_token_stats(jsonl_file):
    token_counts = []
    
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            data = json.loads(line)
            # Извлекаем токены, если ключа нет (для корзины 1) — считаем примерно
            tokens = data.get("token_count", len(data.get("completion", "").split()) * 1.3)
            token_counts.append(int(tokens))
            
    if not token_counts:
        return
        
    total_tales = len(token_counts)
    total_tokens = sum(token_counts)
    avg_tokens = total_tokens / total_tales
    max_tokens = max(token_counts)
    min_tokens = min(token_counts)
    
    print(f"📊 ФИНАЛЬНАЯ СТАТИСТИКА ДАТАСЕТА")
    print("-" * 40)
    print(f"Всего сказок:           {total_tales}")
    print(f"Общий объем токенов:    {total_tokens:,}".replace(',', ' '))
    print(f"Средняя длина сказки:   {avg_tokens:.0f} токенов")
    print(f"Самая короткая сказка:  {min_tokens} токенов")
    print(f"Самая длинная сказка:   {max_tokens} токенов")
    print("-" * 40)

# Вызываем статистику
print_token_stats(final_output_file)

Начинаем финальную сборку и очистку...

Обработка: Корзина 1 (Оригинальные идеальные сказки)...
Обработка: Золотой датасет (Обработанные сказки из корзин 2 и 3)...
----------------------------------------
Сборка завершена!
Целых сказок без ошибок: 4353
Спасенных после склейки: 0
Итоговый чистый файл: dataset/master_dataset_ready.jsonl

📊 ФИНАЛЬНАЯ СТАТИСТИКА ДАТАСЕТА
----------------------------------------
Всего сказок:           4353
Общий объем токенов:    4 159 426
Средняя длина сказки:   956 токенов
Самая короткая сказка:  0 токенов
Самая длинная сказка:   2142 токенов
----------------------------------------


In [4]:
import json
import os

input_file = "dataset/master_dataset_ready.jsonl"
output_file = "dataset/master_dataset_final.jsonl"

valid_tales = 0
empty_tales = 0

print("Начинаем очистку от пустых строк...\n")

if not os.path.exists(input_file):
    print(f"Файл {input_file} не найден!")
else:
    with open(input_file, 'r', encoding='utf-8') as f_in, open(output_file, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
                
            data = json.loads(line)
            text = data.get("completion", "").strip()
            
            # Проверяем, что текст существует и имеет адекватную длину
            if len(text) > 50:
                f_out.write(line + '\n')
                valid_tales += 1
            else:
                empty_tales += 1

    print("-" * 30)
    print(f"Удалено пустых/бракованных сказок: {empty_tales}")
    print(f"Осталось полноценных сказок: {valid_tales}")
    print(f"Идеальный датасет для обучения: {output_file}")

Начинаем очистку от пустых строк...

------------------------------
Удалено пустых/бракованных сказок: 544
Осталось полноценных сказок: 3809
Идеальный датасет для обучения: dataset/master_dataset_final.jsonl


In [5]:
import json
import os

ready_file = "dataset/master_dataset_ready.jsonl"
original_master = "dataset/master_dataset_clean.jsonl"
rescue_file = "dataset/distill/tales_rescue_mission.jsonl"

broken_titles = set()

print("1. Ищем названия зацензуренных сказок...")
with open(ready_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        # Если текста нет или он слишком короткий
        if len(data.get("completion", "").strip()) < 50:
            broken_titles.add(data.get("title", ""))

print(f"Найдено {len(broken_titles)} уникальных сказок для спасения.\n")

print("2. Извлекаем их оригиналы из исходного датасета...")
rescued_count = 0

if not os.path.exists(original_master):
    print(f"Ошибка: Исходный файл {original_master} не найден!")
else:
    with open(original_master, 'r', encoding='utf-8') as f_in, open(rescue_file, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            # Если название есть в нашем списке "отказников"
            if data.get("title", "") in broken_titles:
                f_out.write(line + '\n')
                rescued_count += 1
                # Удаляем из сета, чтобы не дублировать, если в базе есть одинаковые названия
                broken_titles.remove(data.get("title", ""))

    print("-" * 40)
    print(f"Успешно извлечено {rescued_count} оригиналов.")
    print(f"Файл для обработки локальной моделью: {rescue_file}")

1. Ищем названия зацензуренных сказок...
Найдено 544 уникальных сказок для спасения.

2. Извлекаем их оригиналы из исходного датасета...
----------------------------------------
Успешно извлечено 541 оригиналов.
Файл для обработки локальной моделью: dataset/distill/tales_rescue_mission.jsonl


In [2]:
import json
import os
from tqdm import tqdm
from openai import OpenAI
from transformers import AutoTokenizer

# Подключаемся к локальному LM Studio
client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-2.8b")

# Тот самый бескомпромиссный промпт
SYSTEM_PROMPT = """You are an expert folklorist and a conservative editor of authentic 18th-to-19th-century folklore. Your task is to rewrite and logically repair the provided fairy tale.

CRITICAL CONSTRAINT: 
You must be concise. The story MUST be complete and conclusively finished within 1200-1500 words. Do not ramble. 

Follow these rules strictly:
1. NO CENSORSHIP: Preserve all dark, macabre, or violent elements typical of original folklore.
2. DYNAMIC PACING: Do not over-describe scenery or stretch dialogues. Keep the plot moving forward dynamically like a traditional folktale. 
3. LOGICAL REPAIR: Fix any broken plots or sudden jumps, ensuring smooth transitions.
4. TONE: Maintain an archaic, objective narrative tone. 

Output ONLY the story text. End the text logically."""

def process_rescue_mission(input_file, output_file):
    processed_count = 0
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            processed_count = sum(1 for _ in f)
        print(f"Найдено {processed_count} уже спасенных сказок. Продолжаем...")

    total_lines = 0
    with open(input_file, "r", encoding="utf-8") as f:
        total_lines = sum(1 for line in f if line.strip())

    if processed_count >= total_lines:
        print("Все сказки-отказники уже обработаны!")
        return

    print(f"Осталось спасти: {total_lines - processed_count} из {total_lines}")

    with open(input_file, "r", encoding="utf-8") as in_f, open(output_file, "a", encoding="utf-8") as out_f:
        progress_bar = tqdm(total=total_lines, initial=processed_count, desc="Локальная обработка (Qwen)")
        
        for index, line in enumerate(in_f):
            if index < processed_count:
                continue
            
            line = line.strip()
            if not line:
                continue
            
            data = json.loads(line)
            # Извлекаем оригинальный текст, который мы вытащили на предыдущем шаге
            original_text = data.get("completion", "")

            try:
                # Отправляем на локальную видеокарту
                response = client.chat.completions.create(
                    model="local-model",
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": f"Story to condense:\n{original_text}"}
                    ],
                    temperature=0.3,
                    max_tokens=2048 
                )
                
                result_text = response.choices[0].message.content
                data["completion"] = result_text
                
                # Точный подсчет токенов
                data["token_count"] = len(tokenizer.encode(result_text))
                
                out_f.write(json.dumps(data, ensure_ascii=False) + "\n")
                out_f.flush()
                progress_bar.update(1)
                
            except Exception as e:
                print(f"\nОшибка сервера на индексе {index}: {e}")
                print("Скрипт остановлен. Убедись, что сервер LM Studio работает и модель загружена.")
                break
                
        progress_bar.close()
        print("-" * 40)
        print(f"Миссия по спасению завершена!")
        print(f"Спасенные данные лежат здесь: {output_file}")

if __name__ == "__main__":
    INPUT_FILE = "dataset/distill/tales_rescue_mission.jsonl"
    OUTPUT_FILE = "dataset/distill/tales_rescued_processed.jsonl"
    
    process_rescue_mission(INPUT_FILE, OUTPUT_FILE)

Загрузка токенизатора...
Найдено 541 уже спасенных сказок. Продолжаем...
Все сказки-отказники уже обработаны!


In [3]:
import json
import os

final_base_file = "dataset/master_dataset_final.jsonl"
rescued_file = "dataset/distill/tales_rescued_processed.jsonl"
ultimate_file = "dataset/master_dataset_ultimate.jsonl"

valid_lines = 0
fixed_lines = 0
error_lines = 0
empty_lines = 0
token_counts = []

print("Начинаем сборку Абсолютного Датасета...\n")

def process_and_write(input_path, output_handle, source_name):
    global valid_lines, fixed_lines, error_lines, empty_lines, token_counts
    
    if not os.path.exists(input_path):
        print(f"ВНИМАНИЕ: Файл {input_path} не найден!")
        return
    
    print(f"Обработка: {source_name}...")
    with open(input_path, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            
            # Вложенная функция для проверки и сохранения одного JSON-объекта
            def check_and_save(json_str):
                global valid_lines, empty_lines, token_counts
                data = json.loads(json_str)
                text = data.get("completion", "").strip()
                
                # Проверяем наличие контента
                if len(text) > 50:
                    # Перезаписываем корректный JSON (чтобы точно не было мусора)
                    output_handle.write(json.dumps(data, ensure_ascii=False) + '\n')
                    valid_lines += 1
                    
                    # Собираем токены для статистики
                    tokens = data.get("token_count", len(text.split()) * 1.3)
                    token_counts.append(int(tokens))
                else:
                    empty_lines += 1

            # Блок отлова ошибок
            try:
                check_and_save(line)
            except json.JSONDecodeError:
                # Спасаем склейки }{
                if '}{' in line:
                    parts = line.replace('}{', '}\n{').split('\n')
                    for part in parts:
                        try:
                            check_and_save(part)
                            global fixed_lines
                            fixed_lines += 1
                        except json.JSONDecodeError:
                            error_lines += 1
                else:
                    error_lines += 1

# 1. ЗАПУСК СБОРКИ
with open(ultimate_file, 'w', encoding='utf-8') as f_out:
    process_and_write(final_base_file, f_out, "Основная база (без отказников)")
    process_and_write(rescued_file, f_out, "Спасенные от цензуры сказки")

# 2. ВЫВОД ИТОГОВ
print("-" * 40)
print("Сборка завершена успешно!")
print(f"✅ Целых и непустых сказок: {valid_lines}")
print(f"🔧 Спасенных после склейки: {fixed_lines}")
if empty_lines > 0:
    print(f"🗑 Удалено пустых текстов: {empty_lines}")
if error_lines > 0:
    print(f"❌ Невосстановимых строк: {error_lines}")

if token_counts:
    total_tales = len(token_counts)
    total_tokens = sum(token_counts)
    avg_tokens = total_tokens / total_tales
    
    print(f"\n📊 ФИНАЛЬНАЯ СТАТИСТИКА (Токены)")
    print("-" * 40)
    print(f"Всего сказок в датасете:  {total_tales}")
    print(f"Общий объем токенов:      {total_tokens:,}".replace(',', ' '))
    print(f"Средняя длина сказки:     {avg_tokens:.0f}")
    print(f"Самая короткая сказка:    {min(token_counts)}")
    print(f"Самая длинная сказка:     {max(token_counts)}")
    print("-" * 40)
    
print(f"🔥 Твой идеальный монолит сохранен: {ultimate_file}")

Начинаем сборку Абсолютного Датасета...

Обработка: Основная база (без отказников)...
Обработка: Спасенные от цензуры сказки...
----------------------------------------
Сборка завершена успешно!
✅ Целых и непустых сказок: 4350
🔧 Спасенных после склейки: 0

📊 ФИНАЛЬНАЯ СТАТИСТИКА (Токены)
----------------------------------------
Всего сказок в датасете:  4350
Общий объем токенов:      4 662 461
Средняя длина сказки:     1072
Самая короткая сказка:    38
Самая длинная сказка:     2312
----------------------------------------
🔥 Твой идеальный монолит сохранен: dataset/master_dataset_ultimate.jsonl


In [6]:
import json

input_file = "dataset/master_dataset_ultimate.jsonl"
perfect_file = "dataset/master_dataset_perfect.jsonl"
outliers_file = "dataset/distill/tales_outliers.jsonl"

# Ставим лимит 2000, чтобы оставить 48 токенов под системные теги при обучении
TOKEN_LIMIT = 2000

perfect_count = 0
outlier_count = 0

print(f"Ищем сказки длиннее {TOKEN_LIMIT} токенов...\n")

with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(perfect_file, 'w', encoding='utf-8') as f_perf, \
     open(outliers_file, 'w', encoding='utf-8') as f_outl:
    
    for line in f_in:
        line = line.strip()
        if not line:
            continue
            
        data = json.loads(line)
        # Получаем количество токенов
        tokens = int(data.get("token_count", 0))
        
        if tokens <= TOKEN_LIMIT:
            f_perf.write(line + '\n')
            perfect_count += 1
        else:
            f_outl.write(line + '\n')
            outlier_count += 1

print("-" * 30)
print(f"✅ Идеальных сказок (<= {TOKEN_LIMIT} токенов): {perfect_count}")
print(f"⚠️ Выбросов (> {TOKEN_LIMIT} токенов): {outlier_count}")
print("-" * 30)

Ищем сказки длиннее 2000 токенов...

------------------------------
✅ Идеальных сказок (<= 2000 токенов): 4317
⚠️ Выбросов (> 2000 токенов): 33
------------------------------


In [1]:
import pandas as pd
import re
import os

# Пути к файлам
INPUT_FILE = "dataset/master_dataset_clean.jsonl"
OUTPUT_FILE = "dataset/master_dataset_filtered.jsonl"

def clean_fairy_tale(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Удаление мета-тегов в квадратных скобках: [Illustration: ...], [Pg 12]
    text = re.sub(r'\[.*?\]', '', text)
    
    # 2. Удаление сносок в круглых скобках
    text = re.sub(r'\(Note:.*?\)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\(Footnote:.*?\)', '', text, flags=re.IGNORECASE)
    
    # 3. Отсечение Гутенберговских предисловий и послесловий
    markers_to_split = [
        "*** START OF THE PROJECT GUTENBERG",
        "*** END OF THE PROJECT GUTENBERG",
        "End of the Project Gutenberg",
        "FOOTNOTES:"
    ]
    for marker in markers_to_split:
        if marker in text:
            # Если это начало, берем всё, что ПОСЛЕ него
            if "START" in marker:
                text = text.split(marker)[-1]
            # Если это конец, берем всё, что ДО него
            else:
                text = text.split(marker)[0]

    # 4. Удаление лишних пустых строк и множественных пробелов
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    
    return text.strip(" \n\r\t-")

def main():
    print(f"Загрузка датасета: {INPUT_FILE}...")
    if not os.path.exists(INPUT_FILE):
        print(f"Ошибка: Файл {INPUT_FILE} не найден!")
        return

    # Загружаем JSONL в DataFrame
    df = pd.read_json(INPUT_FILE, lines=True)
    initial_count = len(df)
    
    # Проверяем, как называется колонка с текстом сказки
    # Обычно это 'completion', но на всякий случай проверяем и 'text'
    text_col = 'completion' if 'completion' in df.columns else 'text'
    
    if text_col not in df.columns:
        print(f"Ошибка: Не найдена колонка с текстом! Доступные колонки: {df.columns.tolist()}")
        return

    print(f"Очистка колонки '{text_col}'...")
    
    # Применяем функцию очистки ко всем строкам
    df[text_col] = df[text_col].apply(clean_fairy_tale)
    
    # Удаляем строки, которые после очистки оказались пустыми или слишком короткими
    # (например, если сказка состояла только из одного тега или мусора)
    df = df[df[text_col].str.len() > 100]
    
    final_count = len(df)
    dropped_count = initial_count - final_count
    
    print(f"Сохранение результата в {OUTPUT_FILE}...")
    # Сохраняем в новый JSONL файл
    df.to_json(OUTPUT_FILE, orient="records", lines=True, force_ascii=False)
    
    print("\n--- Готово! ---")
    print(f"Обраработано строк: {initial_count}")
    if dropped_count > 0:
        print(f"Удалено пустых/коротких текстов: {dropped_count}")
    print(f"Итоговый размер датасета: {final_count} строк.")

if __name__ == "__main__":
    main()

Загрузка датасета: dataset/master_dataset_clean.jsonl...
Очистка колонки 'completion'...
Сохранение результата в dataset/master_dataset_filtered.jsonl...

--- Готово! ---
Обраработано строк: 4355
Итоговый размер датасета: 4355 строк.
